<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/16%EC%9E%A5_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EB%A5%98/%5B16-1%5D%20%EC%8A%A4%ED%8C%B8%20%ED%96%84%20%EB%B6%84%EB%A5%98%20%EC%8B%A4%EC%8A%B5_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/16%EC%9E%A5_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EB%A5%98/%5B16-1%5D%20%EC%8A%A4%ED%8C%B8%20%ED%96%84%20%EB%B6%84%EB%A5%98%20%EC%8B%A4%EC%8A%B5_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [16-1] 스팸 햄 분류 실습 (수정본)

자연어처리 바이블 16장 실습 [16-1]을 최신 Colab에서 실행되도록 고친 노트북입니다. SMS 문자 5,574개를 **스팸(spam)/햄(ham)**으로 분류하는 다섯 가지 분류기를 학습하고 비교합니다.

**원본 대비 수정 사항**
1. **`import np` → `import numpy as np`**: 원본의 `np`는 NumPy가 아닌 별도 패키지로, NumPy 2.0에서 `np.float_ was removed` 오류가 납니다.
2. **`punkt_tab` 추가**: 최신 NLTK의 `sent_tokenize`/`word_tokenize`에 필요합니다.
3. **데이터 자동 다운로드**: `SMSSpamCollection` 파일을 업로드하지 않아도 GitHub에서 바로 받습니다.
4. **문자 2개가 합쳐지던 문제 수정**: 문자 안의 큰따옴표 때문에 `csv.reader`가 두 줄을 하나로 읽었습니다(5,574 → 5,572개). `quoting=csv.QUOTE_NONE`으로 모든 문자를 그대로 읽습니다.
5. **전처리 순서 수정**: 소문자로 바꾼 뒤 불용어를 제거해, 문장 첫머리의 'The', 'You' 같은 불용어도 제거되게 했습니다.
6. **평가 데이터 1개 누락 수정**: 원본은 `trainset_size+1`부터 평가 데이터를 잘라 1개가 빠졌습니다.
7. **정리**: 다섯 번 반복되던 평가 코드를 반복문으로 묶고, `random_state`로 결과를 고정했습니다. 결과 비교표와 새 문자 예측 셀을 추가했습니다.

※ 4~6번 수정으로 데이터가 조금 달라져 결과 숫자가 교재 출력과 약간 다릅니다.

## 1. 환경 설정

In [1]:
import csv
import urllib.request

import numpy as np        # 수정: 원본 'import np'(별도 패키지) 대신 NumPy
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:   # 수정: punkt_tab 추가
    nltk.download(pkg, quiet=True)

## 2. 데이터 불러오기

`SMSSpamCollection`은 한 줄에 `라벨<탭>문자` 형식입니다 (라벨: `ham` 또는 `spam`).

In [2]:
# 수정: 파일 업로드 대신 GitHub에서 바로 내려받기
URL = ('https://raw.githubusercontent.com/mrdantzig/nlp-bible-code/master/'
       '16%EC%9E%A5_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EB%A5%98/SMSSpamCollection')
urllib.request.urlretrieve(URL, 'SMSSpamCollection')

sms_labels, sms_raw = [], []
with open('SMSSpamCollection', encoding='utf8') as smsdata:
    # 수정: QUOTE_NONE — 문자 속 큰따옴표 때문에 두 줄이 합쳐지지 않도록
    for line in csv.reader(smsdata, delimiter='\t', quoting=csv.QUOTE_NONE):
        sms_labels.append(line[0])
        sms_raw.append(line[1])

print('문자 수:', len(sms_raw))                       # 5574
print(pd.Series(sms_labels).value_counts())          # ham 4827, spam 747
for label, text in list(zip(sms_labels, sms_raw))[:3]:
    print(label, ':', text)

문자 수: 5574
ham     4827
spam     747
Name: count, dtype: int64
ham : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham : Ok lar... Joking wif u oni...
spam : Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


## 3. 전처리

토큰화 → 소문자 → 불용어 제거 → 3글자 미만 제거 → 표제어 추출

In [3]:
stop = set(stopwords.words('english'))
lmtzr = WordNetLemmatizer()


def preprocessing(text):
    # tokenize into words
    tokens = [word for sent in nltk.sent_tokenize(text) for word in nltk.word_tokenize(sent)]
    # lower capitalization  (수정: 불용어 제거보다 먼저)
    tokens = [word.lower() for word in tokens]
    # remove stopwords
    tokens = [token for token in tokens if token not in stop]
    # remove words less than three letters
    tokens = [word for word in tokens if len(word) >= 3]
    # lemmatize
    tokens = [lmtzr.lemmatize(word) for word in tokens]
    return ' '.join(tokens)


sms_data = [preprocessing(text) for text in sms_raw]
print(sms_raw[0])
print('→', sms_data[0])

Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
→ jurong point crazy available bugis great world buffet ... cine got amore wat ...


## 4. 학습/평가 데이터 나누기와 TF-IDF 특징값 추출

원본처럼 앞 70%를 학습, 나머지 30%를 평가에 씁니다.

In [4]:
trainset_size = int(round(len(sms_data) * 0.70))   # Split Train data and Test data
print('The training set size for this classifier is', trainset_size)

x_train = np.array(sms_data[:trainset_size])
y_train = np.array(sms_labels[:trainset_size])
x_test = np.array(sms_data[trainset_size:])      # 수정: 원본은 trainset_size+1부터 → 1개 누락
y_test = np.array(sms_labels[trainset_size:])
print('train:', len(x_train), ' test:', len(x_test))

# TF-IDF vectorizer (unigram + bigram)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer2 = TfidfVectorizer(min_df=2, ngram_range=(1, 2), stop_words='english',
                              strip_accents='unicode', norm='l2')
X_train = vectorizer2.fit_transform(x_train)
X_test = vectorizer2.transform(x_test)
print('특징 수:', X_train.shape[1])

The training set size for this classifier is 3902
train: 3902  test: 1672
특징 수: 5868


## 5. 다섯 가지 분류기 학습

Naive Bayes, Decision tree, Stochastic gradient descent, Support Vector Machines, Random forest

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# 수정: random_state로 실행할 때마다 같은 결과가 나오게 고정
models = {
    'NB':  MultinomialNB(),
    'DT':  DecisionTreeClassifier(random_state=0),
    'SGD': SGDClassifier(alpha=.0001, random_state=0),
    'SVM': LinearSVC(),
    'RFA': RandomForestClassifier(n_estimators=10, random_state=0),
}
predictions = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    predictions[name] = clf.predict(X_test)
print('학습 완료:', list(models))

학습 완료: ['NB', 'DT', 'SGD', 'SVM', 'RFA']


## 6. 평가: 혼동 행렬과 분류 리포트

혼동 행렬은 `[[햄→햄, 햄→스팸], [스팸→햄, 스팸→스팸]]` 순서입니다.

In [6]:
from sklearn.metrics import confusion_matrix, classification_report

for name, y_pred in predictions.items():
    print(f'\n confusion_matrix {name} \n')
    print(confusion_matrix(y_test, y_pred))
    print('\n Here is the classification report:')
    print(classification_report(y_test, y_pred, digits=3))


 confusion_matrix NB 

[[1444    0]
 [  52  176]]

 Here is the classification report:
              precision    recall  f1-score   support

         ham      0.965     1.000     0.982      1444
        spam      1.000     0.772     0.871       228

    accuracy                          0.969      1672
   macro avg      0.983     0.886     0.927      1672
weighted avg      0.970     0.969     0.967      1672


 confusion_matrix DT 

[[1425   19]
 [  38  190]]

 Here is the classification report:
              precision    recall  f1-score   support

         ham      0.974     0.987     0.980      1444
        spam      0.909     0.833     0.870       228

    accuracy                          0.966      1672
   macro avg      0.942     0.910     0.925      1672
weighted avg      0.965     0.966     0.965      1672


 confusion_matrix SGD 

[[1438    6]
 [  21  207]]

 Here is the classification report:
              precision    recall  f1-score   support

         ham      0.986   

## 7. (추가) 결과 한눈에 비교

평가 데이터의 대부분이 햄이라 정확도는 모두 높게 나옵니다. **스팸을 얼마나 놓쳤는지(스팸 재현율)**와 **스팸 F1**을 함께 봐야 차이가 드러납니다.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rows = []
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=['ham', 'spam'])
    rows.append({'분류기': name,
                 '스팸 오탐(햄→스팸)': cm[0, 1],
                 '놓친 스팸(스팸→햄)': cm[1, 0],
                 '정확도': accuracy_score(y_test, y_pred),
                 '스팸 정밀도': precision_score(y_test, y_pred, pos_label='spam'),
                 '스팸 재현율': recall_score(y_test, y_pred, pos_label='spam'),
                 '스팸 F1': f1_score(y_test, y_pred, pos_label='spam')})
pd.DataFrame(rows).set_index('분류기').round(3)

,스팸 오탐(햄→스팸),놓친 스팸(스팸→햄),정확도,스팸 정밀도,스팸 재현율,스팸 F1
분류기,,,,,,
NB,0,52,0.969,1.000,0.772,0.871
DT,19,38,0.966,0.909,0.833,0.870
SGD,6,21,0.984,0.972,0.908,0.939
SVM,6,23,0.983,0.972,0.899,0.934
RFA,2,50,0.969,0.989,0.781,0.873


## 8. (추가) 새 문자 분류해 보기

아래 `samples`에 직접 문장을 넣어 보세요.

세 번째 문자(계정 해지를 미끼로 비밀번호를 요구하는 피싱)는 두 모델 모두 햄으로 판단합니다. 이 데이터의 스팸은 대부분 '당첨·무료·전화' 유형이라, 학습 데이터에 드문 유형은 놓칠 수 있습니다.

In [8]:
samples = [
    'Congratulations! You have won a free ticket. Call now to claim your prize.',
    'Are we still meeting for lunch tomorrow?',
    'URGENT: your account will be closed. Reply with your password to keep it active.',
    'Hi,we’ve attached your invoice. Thanks again! 2026-10-24-Support Desk Contact: +1(813) 934-567'
]
X_new = vectorizer2.transform([preprocessing(s) for s in samples])
for s, nb, svm in zip(samples, models['NB'].predict(X_new), models['SVM'].predict(X_new)):
    print(f'NB: {nb:4s} | SVM: {svm:4s} | {s}')

NB: spam | SVM: spam | Congratulations! You have won a free ticket. Call now to claim your prize.
NB: ham  | SVM: ham  | Are we still meeting for lunch tomorrow?
NB: ham  | SVM: ham  | URGENT: your account will be closed. Reply with your password to keep it active.
NB: ham  | SVM: ham  | Hi,we’ve attached your invoice. Thanks again! 2026-10-24-Support Desk Contact: +1(813) 934-567
